# 📚 Seminário — Capítulo 8: Busca Semântica e RAG## Caderno de Exercícios Práticos — **GABARITO** 🔑**Disciplina:** Recuperação de Informação / NLP  **Baseado em:** Capítulo 8 — *Busca Semântica e Recuperação-Geração Aumentada*  ---### ObjetivoEste notebook acompanha a apresentação do Capítulo 8 e propõe exercícios práticos de **dificuldade progressiva** sobre:1. **Recuperação Densa** — embeddings e busca por similaridade  2. **Reclassificação (Reranking)** — reordenação de resultados por relevância  3. **RAG (Retrieval-Augmented Generation)** — geração fundamentada em documentos recuperados  4. **Métricas de Avaliação** — MAP, Precision@k  > ⚠️ **Instruções:** Este é o **gabarito completo** com todas as soluções.

---## 🔧 Configuração do AmbienteExecute a célula abaixo para instalar as dependências necessárias.

In [ ]:
# Instalação das dependências
!pip install -q cohere numpy pandas scikit-learn rank_bm25 faiss-cpu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 54.5 MB/s eta 0:00:00


### Configuração da APIObtenha sua chave de API gratuita em [dashboard.cohere.com](https://dashboard.cohere.com/).  Cole-a na célula abaixo.

In [ ]:
# Insira sua chave de API do Cohere abaixo
api_key = ''  # <-- Insira sua chave aqui

import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm

co = cohere.Client(api_key)
print("✅ Cliente Cohere configurado com sucesso!" if api_key else "❌ Insira sua chave de API!")

✅ Cliente Cohere configurado com sucesso!


---## 📘 Parte 1 — Recuperação Densa (Dense Retrieval)### ConceitoA **recuperação densa** transforma textos em vetores numéricos (embeddings) e encontra os documentos mais similares a uma consulta comparando distâncias no espaço vetorial.> 💡 Diferentemente da busca por palavras-chave, a busca semântica encontra resultados **por significado**, mesmo que não compartilhem palavras em comum com a consulta.

### Exercício 1 — Preparação dos Dados (Fácil ⭐)O texto abaixo é um trecho da página da Wikipédia sobre o filme *Interstellar*.  **Tarefa:** Divida o texto em frases (usando o ponto `.` como separador) e remova espaços/quebras de linha extras.

In [ ]:
# Corpus: trecho da Wikipédia sobre Interstellar
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and target $773 million with subsequent re-releases), making it the tenth-highest-grossing film of 2014.
It has been praised for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It also received praise from many astronomers for its scientific accuracy and target depiction of theoretical astrophysics.
Since its premiere, Interstellar has gained a cult following, and is now regarded by many sci-fi experts as one of the best science fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades.
"""

print(f"Texto completo tem {len(text)} caracteres.")

Texto completo tem 1942 caracteres.


In [ ]:
# Dividir o texto em frases usando split('.') e limpar espaços extras
texts = text.split('.')
texts = [t.strip(' \n') for t in texts if t.strip(' \n')]

print(f"Total de frases: {len(texts)}")
print(f"Primeira frase: {texts[0][:80]}...")

Total de frases: 15
Primeira frase: Interstellar is a 2014 epic science fiction film co-written, directed, and produ...


### Exercício 2 — Geração de Embeddings (Fácil ⭐)Agora vamos converter cada frase em um vetor numérico usando o modelo de embeddings do Cohere.**Tarefa:** Use `co.embed()` para gerar os embeddings das frases.  - `texts`: lista de textos  - `input_type`: `"search_document"` (pois estamos indexando documentos)

In [ ]:
# Gerar embeddings usando co.embed()
response = co.embed(
    texts=texts,
    input_type="search_document",
)

embeddings = np.array(response.embeddings)
print(f"Shape dos embeddings: {embeddings.shape}")
print(f"Cada frase é representada por um vetor de {embeddings.shape[1]} dimensões.")

Shape dos embeddings: (15, 4096)
Cada frase é representada por um vetor de 4096 dimensões.


### Exercício 3 — Construção do Índice de Busca (Médio ⭐⭐)Para buscar eficientemente, usamos a biblioteca **FAISS** (Facebook AI Similarity Search).**Tarefa:** Crie um índice FAISS do tipo `IndexFlatL2` (busca exata por distância L2) e adicione os embeddings.

In [ ]:
import faiss

# Criar o índice FAISS
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeddings))

print(f"Índice criado. Treinado: {index.is_trained}")
print(f"Vetores no índice: {index.ntotal}")

Índice criado. Treinado: True
Vetores no índice: 15


### Exercício 4 — Função de Busca Semântica (Médio ⭐⭐)**Tarefa:** Complete a função `search()` que:1. Gera o embedding da consulta (com `input_type="search_query"`)2. Busca os vizinhos mais próximos no índice FAISS3. Retorna os resultados formatados em um DataFrame

In [ ]:
def search(query, num_results=3):
    """Realiza busca semântica usando embeddings e FAISS."""

    # 1. Gerar o embedding da consulta
    query_embed = co.embed(
        texts=[query],
        input_type="search_query",
    ).embeddings[0]

    # 2. Buscar os vizinhos mais próximos no índice
    distances, similar_items = index.search(
        np.float32([query_embed]), num_results
    )

    # 3. Formatar os resultados
    texts_np = np.array(texts)
    results = pd.DataFrame({
        'textos': texts_np[similar_items[0]],
        'distância': distances[0]
    })

    print(f"Consulta: '{query}'\nVizinhos mais próximos:")
    return results

# Teste a função
results = search("How accurate was the science?")
results

Consulta: 'How accurate was the science?'
Vizinhos mais próximos:


,textos,distância
0,It also received praise from many astronomers ...,7244.883789
1,Cinematographer Hoyte van Hoytema shot it on 3...,11792.724609
2,Interstellar uses extensive practical and mini...,11814.633789


### Exercício 5 — Comparação: Busca Semântica vs. Busca por Palavras-chave (Médio ⭐⭐)Vamos implementar a busca por palavras-chave com **BM25** para comparar com a busca semântica.**Tarefa:** Complete o tokenizador e a função de busca por palavras-chave.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    """Tokeniza o texto para o BM25: lowercase, remove pontuação e stopwords."""
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

# Tokenizar o corpus
tokenized_corpus = [bm25_tokenizer(passage) for passage in tqdm(texts)]
bm25 = BM25Okapi(tokenized_corpus)

def keyword_search(query, top_k=3):
    """Busca por palavras-chave usando BM25."""
    print("Consulta:", query)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))

    top_n = np.argsort(bm25_scores)[::-1]

    print(f"\nTop {top_k} resultados (BM25):")
    for idx in top_n[:top_k]:
        print(f"  Score: {bm25_scores[idx]:.3f} | {texts[idx][:100]}...")

    return top_n[:top_k]

# Comparação de resultados
print("=" * 60)
print("BUSCA SEMÂNTICA:")
print("=" * 60)
search("How accurate was the science?")

print("\n" + "=" * 60)
print("BUSCA POR PALAVRAS-CHAVE (BM25):")
print("=" * 60)
keyword_search("How accurate was the science?")

100%|██████████| 15/15 [00:00<00:00, 54613.33it/s]

BUSCA SEMÂNTICA:
Consulta: 'How accurate was the science?'
Vizinhos mais próximos:

BUSCA POR PALAVRAS-CHAVE (BM25):
Consulta: How accurate was the science?

Top 3 resultados (BM25):
  Score: 1.353 | Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher N...
  Score: 1.258 | Since its premiere, Interstellar has gained a cult following, and is now regarded by many sci-fi exp...
  Score: 1.039 | Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executive produce...


array([ 0, 13,  4])

### 📝 Questão Reflexiva 1 (Fácil ⭐)No espaço abaixo, responda:**Por que a busca semântica encontrou o resultado correto ("received praise from many astronomers for its scientific accuracy...") enquanto a busca BM25 não conseguiu?**

In [ ]:
# Resposta:
# A busca semântica captura o SIGNIFICADO da consulta, não apenas as palavras exatas.
# A consulta "How accurate was the science?" não compartilha muitas palavras-chave
# com a resposta correta, que fala sobre "scientific accuracy" e "theoretical astrophysics".
# O BM25, por ser uma busca lexical, prioriza correspondência exata de termos,
# retornando resultados que contêm "science" mas não necessariamente respondem à pergunta.
# Já os embeddings colocam a consulta e a resposta correta próximas no espaço vetorial
# porque possuem significado similar, mesmo com palavras diferentes.

---## 📗 Parte 2 — Reclassificação (Reranking)### ConceitoO **reclassificador** é um modelo que recebe uma consulta e um conjunto de resultados candidatos e atribui uma **pontuação de relevância** a cada resultado. Funciona como um *cross-encoder* — analisa a consulta e cada documento simultaneamente.> 💡 Na prática, o reranking é usado como **segunda etapa** de um pipeline de busca: primeiro recuperamos candidatos (com BM25, embeddings ou híbrido), depois refinamos a ordem com o reclassificador.

### Exercício 6 — Reclassificação com Cohere (Médio ⭐⭐)**Tarefa:** Use `co.rerank()` para reclassificar os textos do corpus em relação a uma consulta.  Parâmetros:- `query`: a consulta- `documents`: lista de documentos- `top_n`: número de resultados- `return_documents`: `True`

In [ ]:
query = "How accurate was the science?"

# Reclassificar os textos usando Cohere Rerank
results = co.rerank(
    query=query,
    documents=texts,
    top_n=3,
    return_documents=True,
)

# Exibir resultados
print(f"Consulta: '{query}'\n")
for idx, result in enumerate(results.results):
    print(f"{idx+1}. [Relevância: {result.relevance_score:.4f}]")
    print(f"   {result.document.text[:120]}...")
    print()

Consulta: 'How accurate was the science?'

1. [Relevância: 0.1972]
   It also received praise from many astronomers for its scientific accuracy and target depiction of theoretical astrophysi...

2. [Relevância: 0.0385]
   Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel thr...

3. [Relevância: 0.0371]
   The film had a worldwide gross over $677 million (and target $773 million with subsequent re-releases), making it the te...



### Exercício 7 — Pipeline Completo: BM25 + Reranking (Difícil ⭐⭐⭐)Agora vamos combinar as duas abordagens: primeiro recuperamos candidatos com BM25, depois refinamos com reranking.**Tarefa:** Complete a função que:1. Recupera os top `num_candidates` resultados com BM252. Passa esses candidatos para o reclassificador do Cohere3. Retorna os top `top_k` resultados reclassificados

In [ ]:
def keyword_search_and_rerank(query, top_k=3, num_candidates=10):
    """Pipeline de busca em duas etapas: BM25 + Reranking."""
    print(f"Consulta: '{query}'\n")

    # Etapa 1: Busca BM25 (primeira etapa)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argsort(bm25_scores)[::-1][:num_candidates]

    print(f"--- Etapa 1: Top 3 do BM25 (de {num_candidates} candidatos) ---")
    for idx in top_n[:3]:
        print(f"  Score: {bm25_scores[idx]:.3f} | {texts[idx][:90]}...")

    # Etapa 2: Reclassificação (segunda etapa)
    candidate_docs = [texts[idx] for idx in top_n]

    reranked = co.rerank(
        query=query,
        documents=candidate_docs,
        top_n=top_k,
        return_documents=True,
    )

    print(f"\n--- Etapa 2: Top {top_k} após Reranking ---")
    for idx, hit in enumerate(reranked.results):
        print(f"  {idx+1}. [Relevância: {hit.relevance_score:.4f}]")
        print(f"     {hit.document.text[:90]}...")

    return reranked

# Teste o pipeline completo
reranked_results = keyword_search_and_rerank("How accurate was the science?")

Consulta: 'How accurate was the science?'

--- Etapa 1: Top 3 do BM25 (de 10 candidatos) ---
  Score: 1.353 | Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Chr...
  Score: 1.258 | Since its premiere, Interstellar has gained a cult following, and is now regarded by many ...
  Score: 1.039 | Caltech theoretical physicist and 2017 Nobel laureate in Physics Kip Thorne was an executi...

--- Etapa 2: Top 3 após Reranking ---
  1. [Relevância: 0.1972]
     It also received praise from many astronomers for its scientific accuracy and target depic...
  2. [Relevância: 0.0371]
     The film had a worldwide gross over $677 million (and target $773 million with subsequent ...
  3. [Relevância: 0.0365]
     Since its premiere, Interstellar has gained a cult following, and is now regarded by many ...


---## 📕 Parte 3 — RAG (Retrieval-Augmented Generation)### Conceito**RAG** combina busca + geração: primeiro recuperamos documentos relevantes, depois apresentamos esses documentos junto com a pergunta a um LLM, que gera uma resposta **fundamentada** nas fontes.> 💡 RAG é a principal técnica para reduzir **alucinações** de LLMs, pois a resposta é baseada em informações recuperadas e não apenas no conhecimento interno do modelo.

### Exercício 8 — Sistema RAG Completo (Difícil ⭐⭐⭐)**Tarefa:** Construa um sistema RAG que:1. Usa a função `search()` para recuperar documentos relevantes2. Passa os documentos para `co.chat()` como contexto3. Gera uma resposta fundamentada com citações

In [ ]:
def rag(query):
    """Sistema RAG: Busca + Geração Fundamentada."""
    print(f"🔍 Pergunta: {query}\n")

    # Etapa 1: Recuperação
    results = search(query)

    # Converter resultados em lista de dicionários
    doc_dicts = [{'text': text} for text in results['textos']]

    # Etapa 2: Geração Fundamentada
    response = co.chat(
        message=query,
        documents=doc_dicts,
    )

    print(f"\n💬 Resposta: {response.text}")

    # Exibir citações (se disponíveis)
    if hasattr(response, 'citations') and response.citations:
        print(f"\n📎 Citações:")
        for citation in response.citations:
            print(f"   - {citation}")

    return response

# Teste o sistema RAG
rag("How accurate was the science?")
print("\n" + "="*60)
rag("What revenue did the movie generate?")

🔍 Pergunta: How accurate was the science?

Consulta: 'How accurate was the science?'
Vizinhos mais próximos:

💬 Resposta: The science in *Interstellar* was accurate and received praise from many astronomers.

📎 Citações:
   - start=34 end=42 text='accurate' document_ids=['doc_0'] type='TEXT_CONTENT'
   - start=56 end=85 text='praise from many astronomers.' document_ids=['doc_0'] type='TEXT_CONTENT'

🔍 Pergunta: What revenue did the movie generate?

Consulta: 'What revenue did the movie generate?'
Vizinhos mais próximos:

💬 Resposta: The movie generated a worldwide gross of over $677 million.

📎 Citações:
   - start=22 end=59 text='worldwide gross of over $677 million.' document_ids=['doc_0'] type='TEXT_CONTENT'


NonStreamedChatResponse(text='The movie generated a worldwide gross of over $677 million.', generation_id='2241761c-bbf5-4ce4-a285-61ac22247c27', response_id='a407b90a-9704-4c44-9ab6-8f956fcdb02d', citations=[ChatCitation(start=22, end=59, text='worldwide gross of over $677 million.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_0', 'text': 'The film had a worldwide gross over $677 million (and target $773 million with subsequent re-releases), making it the tenth-highest-grossing film of 2014'}], is_search_required=None, search_queries=None, search_results=None, finish_reason='COMPLETE', tool_calls=None, chat_history=[UserMessage(role='USER', message='What revenue did the movie generate?', tool_calls=None), ChatbotMessage(role='CHATBOT', message='The movie generated a worldwide gross of over $677 million.', tool_calls=None)], meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images

---## 📙 Parte 4 — Métricas de Avaliação de Recuperação### ConceitoPara avaliar sistemas de busca, usamos métricas como **Precision@k** e **MAP (Mean Average Precision)**.- **Precision@k**: proporção de resultados relevantes nos k primeiros resultados- **Average Precision (AP)**: média das Precision@k calculadas apenas nas posições com resultados relevantes- **MAP**: média das AP sobre todas as consultas do conjunto de teste

### Exercício 9 — Implementação do Precision@k (Médio ⭐⭐)**Tarefa:** Implemente a função `precision_at_k` que calcula a proporção de resultados relevantes nos primeiros k resultados.

In [ ]:
def precision_at_k(retrieved, relevant, k):
    """
    Calcula Precision@k.

    Args:
        retrieved: lista ordenada dos IDs dos documentos retornados
        relevant: conjunto (set) dos IDs dos documentos relevantes
        k: posição de corte

    Returns:
        float: Precision@k
    """
    top_k = retrieved[:k]
    relevant_count = sum(1 for doc in top_k if doc in relevant)
    return relevant_count / k

# Testes
retrieved_1 = ['doc_3', 'doc_1', 'doc_5']
relevant_1 = {'doc_3', 'doc_5'}

print(f"Precision@1: {precision_at_k(retrieved_1, relevant_1, 1):.2f}")  # Esperado: 1.00
print(f"Precision@2: {precision_at_k(retrieved_1, relevant_1, 2):.2f}")  # Esperado: 0.50
print(f"Precision@3: {precision_at_k(retrieved_1, relevant_1, 3):.2f}")  # Esperado: 0.67

Precision@1: 1.00
Precision@2: 0.50
Precision@3: 0.67


### Exercício 10 — Implementação do MAP (Difícil ⭐⭐⭐)**Tarefa:** Implemente as funções `average_precision` e `mean_average_precision`.Lembre-se:- **AP** = média das Precision@k apenas nas posições onde houve resultado relevante, dividida pelo número total de documentos relevantes- **MAP** = média das AP sobre todas as consultas

In [ ]:
def average_precision(retrieved, relevant):
    """
    Calcula Average Precision (AP) para uma única consulta.

    Args:
        retrieved: lista ordenada dos IDs dos documentos retornados
        relevant: conjunto dos IDs dos documentos relevantes

    Returns:
        float: Average Precision
    """
    score = 0.0
    for k in range(1, len(retrieved) + 1):
        if retrieved[k - 1] in relevant:
            score += precision_at_k(retrieved, relevant, k)

    return score / len(relevant) if relevant else 0.0


def mean_average_precision(queries_results):
    """
    Calcula MAP sobre múltiplas consultas.

    Args:
        queries_results: lista de tuplas (retrieved, relevant)

    Returns:
        float: MAP
    """
    aps = [average_precision(ret, rel) for ret, rel in queries_results]
    return np.mean(aps)


# Testes
sys1_q1 = ['doc_3', 'doc_1', 'doc_5']
sys2_q1 = ['doc_1', 'doc_4', 'doc_3']
rel_q1 = {'doc_3', 'doc_5'}

sys1_q2 = ['doc_2', 'doc_6', 'doc_7']
sys2_q2 = ['doc_6', 'doc_7', 'doc_2']
rel_q2 = {'doc_2'}

sys1_q3 = ['doc_8', 'doc_9', 'doc_4']
sys2_q3 = ['doc_4', 'doc_8', 'doc_9']
rel_q3 = {'doc_4'}

print("--- Average Precision por consulta ---")
print(f"AP(Sys1, Q1) = {average_precision(sys1_q1, rel_q1):.2f}")
print(f"AP(Sys1, Q2) = {average_precision(sys1_q2, rel_q2):.2f}")
print(f"AP(Sys1, Q3) = {average_precision(sys1_q3, rel_q3):.2f}")

print(f"\n--- MAP ---")
map_sys1 = mean_average_precision([(sys1_q1, rel_q1), (sys1_q2, rel_q2), (sys1_q3, rel_q3)])
map_sys2 = mean_average_precision([(sys2_q1, rel_q1), (sys2_q2, rel_q2), (sys2_q3, rel_q3)])
print(f"MAP Sistema 1: {map_sys1:.2f}")
print(f"MAP Sistema 2: {map_sys2:.2f}")
print(f"\n{'Sistema 1 é melhor!' if map_sys1 > map_sys2 else 'Sistema 2 é melhor!'}")

--- Average Precision por consulta ---


NameError: name 'precision_at_k' is not defined

---## 🏆 Parte 5 — Desafio Final### Exercício 11 — Pipeline RAG Avançado (Muito Difícil ⭐⭐⭐⭐)Construa um **pipeline RAG completo** que combine todas as técnicas do capítulo:1. Recuperação densa (embeddings + FAISS)2. Reclassificação dos resultados3. Geração fundamentada com citações**Bônus:** Adicione uma etapa de reescrita de consulta antes da busca.

In [ ]:
def advanced_rag_pipeline(user_question, top_retrieval=10, top_rerank=3):
    """
    Pipeline RAG avançado com 3 etapas:
    1. Recuperação densa (FAISS)
    2. Reclassificação (Cohere Rerank)
    3. Geração fundamentada (Cohere Chat)
    """
    print(f"{'='*60}")
    print(f"🧠 Pipeline RAG Avançado")
    print(f"{'='*60}")
    print(f"❓ Pergunta: {user_question}\n")

    # ═══════════════════════════════════════════
    # ETAPA 1: Recuperação Densa
    # ═══════════════════════════════════════════
    print("📥 Etapa 1: Recuperação Densa...")

    retrieval_results = search(user_question, num_results=top_retrieval)

    retrieved_texts = list(retrieval_results['textos'])

    print(f"   → {len(retrieved_texts)} documentos recuperados\n")

    # ═══════════════════════════════════════════
    # ETAPA 2: Reclassificação
    # ═══════════════════════════════════════════
    print("🔄 Etapa 2: Reclassificação...")

    reranked = co.rerank(
        query=user_question,
        documents=retrieved_texts,
        top_n=top_rerank,
        return_documents=True,
    )

    reranked_texts = [hit.document.text for hit in reranked.results]
    print(f"   → Top {top_rerank} documentos reclassificados:")
    for i, hit in enumerate(reranked.results):
        print(f"     {i+1}. [{hit.relevance_score:.4f}] {hit.document.text[:80]}...")
    print()

    # ═══════════════════════════════════════════
    # ETAPA 3: Geração Fundamentada
    # ═══════════════════════════════════════════
    print("💬 Etapa 3: Geração Fundamentada...")

    doc_dicts = [{'text': text} for text in reranked_texts]
    response = co.chat(
        message=user_question,
        documents=doc_dicts,
    )

    print(f"\n{'='*60}")
    print(f"📝 RESPOSTA FINAL:")
    print(f"{'='*60}")
    print(response.text)

    if hasattr(response, 'citations') and response.citations:
        print(f"\n📎 Fontes citadas: {len(response.citations)} citação(ões)")

    return response

# Teste com várias perguntas
advanced_rag_pipeline("How accurate was the science in the movie?")
print("\n\n")
advanced_rag_pipeline("When and where was the movie released?")
print("\n\n")
advanced_rag_pipeline("What awards did the movie win?")

🧠 Pipeline RAG Avançado
❓ Pergunta: How accurate was the science in the movie?

📥 Etapa 1: Recuperação Densa...
Consulta: 'How accurate was the science in the movie?'
Vizinhos mais próximos:
   → 10 documentos recuperados

🔄 Etapa 2: Reclassificação...
   → Top 3 documentos reclassificados:
     1. [0.2738] It also received praise from many astronomers for its scientific accuracy and ta...
     2. [0.0911] Since its premiere, Interstellar has gained a cult following, and is now regarde...
     3. [0.0870] Set in a dystopian future where humanity is struggling to survive, the film foll...

💬 Etapa 3: Geração Fundamentada...

📝 RESPOSTA FINAL:
The film Interstellar received praise from many astronomers for its scientific accuracy and target depiction of theoretical astrophysics.

📎 Fontes citadas: 2 citação(ões)



🧠 Pipeline RAG Avançado
❓ Pergunta: When and where was the movie released?

📥 Etapa 1: Recuperação Densa...
Consulta: 'When and where was the movie released?'
Vizinhos mais pr

NonStreamedChatResponse(text='The movie Interstellar won Best Visual Effects at the 87th Academy Awards.', generation_id='554d1732-e903-4db1-9568-60299ee2485b', response_id='3515cd5c-2c94-47c4-988b-5a7ff4fa646c', citations=[ChatCitation(start=10, end=22, text='Interstellar', document_ids=['doc_0', 'doc_1', 'doc_2'], type='TEXT_CONTENT'), ChatCitation(start=27, end=46, text='Best Visual Effects', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=54, end=74, text='87th Academy Awards.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_0', 'text': 'Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades'}, {'id': 'doc_1', 'text': 'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'}, {'id': 'doc_2', 'text': 'Since its premiere, Interstellar has gained a cult following, and is now regarded by many sci-fi experts as one o

---## 📝 Questão Reflexiva Final (Difícil ⭐⭐⭐)Responda às perguntas abaixo com base no que aprendeu:1. **Quais são as vantagens de usar busca híbrida (semântica + keyword) em vez de apenas uma das abordagens?**2. **Em que cenários o RAG é especialmente útil em comparação com um LLM sem recuperação?**3. **Por que a reclassificação melhora os resultados da busca, e por que não usá-la diretamente sobre todo o corpus?**

In [ ]:
# RESPOSTAS:

# 1. Busca Híbrida:
# A busca semântica captura significado e encontra resultados mesmo sem correspondência
# exata de termos, mas pode falhar em buscas por frases específicas ou nomes próprios.
# A busca por palavras-chave (BM25) é excelente para correspondência exata, mas não
# entende sinônimos ou paráfrases. A combinação das duas oferece cobertura mais
# completa, reduzindo os pontos fracos de cada abordagem individual.

# 2. RAG vs LLM puro:
# RAG é essencial quando: (a) precisamos de informações atualizadas que não estão no
# treinamento do modelo; (b) queremos respostas verificáveis com citações de fontes;
# (c) trabalhamos com dados proprietários ou internos; (d) precisamos reduzir
# alucinações fundamentando as respostas em documentos reais.

# 3. Reclassificação:
# O reclassificador (cross-encoder) analisa consulta e documento simultaneamente,
# capturando interações mais ricas entre eles. Porém, esse processo é computacionalmente
# caro (O(n) chamadas ao modelo). Por isso, é inviável aplicá-lo diretamente sobre
# milhões de documentos. A solução é usar um sistema de recuperação rápido (BM25 ou
# embeddings) como primeira etapa para filtrar candidatos, e então aplicar o
# reclassificador apenas sobre os top-k candidatos.

---## ✅ ConclusãoParabéns por completar o caderno! Você praticou:| Conceito | Técnica | Ferramenta ||----------|---------|------------|| Recuperação Densa | Embeddings + FAISS | `co.embed()` + `faiss` || Busca por Palavras-chave | BM25 | `rank_bm25` || Reclassificação | Cross-encoder | `co.rerank()` || RAG | Busca + Geração | `co.chat()` || Avaliação | MAP, Precision@k | Implementação manual |> 📖 Para aprofundamento, consulte o **Capítulo 8** completo e explore os links para as bibliotecas mencionadas.---*Caderno desenvolvido como material complementar ao Seminário do Capítulo 8 — Busca Semântica e RAG.*